Create New Train Env

In [ ]:
%pip install "gymnasium[box2d]" stable-baselines3 matplotlib pandas
# Installs all required packages:
#   gymnasium[box2d] — the physics simulation environments (BipedalWalker lives here)
#   stable-baselines3 — pre-built RL algorithms including PPO
#   matplotlib — used to graph training rewards
#   pandas — used to load and manipulate the monitor log CSV files

import os                  # Used to build folder paths and create directories
import glob                # Used to find all monitor CSV files by pattern
import numpy as np         # Numerical arrays (used internally by gym and SB3)
import pandas as pd        # DataFrame operations for loading and smoothing reward logs
import matplotlib.pyplot as plt  # Plotting library for all training graphs

import gymnasium as gym    # The environment library — provides BipedalWalker-v3

from stable_baselines3 import PPO  # PPO (Proximal Policy Optimization) — the RL algorithm we train with
from stable_baselines3.common.monitor import Monitor  # Wraps an env to record episode rewards/lengths to CSV
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback, CallbackList
# EvalCallback      — periodically tests the agent and saves the best version
# CheckpointCallback — saves a model snapshot every N steps
# CallbackList       — combines multiple callbacks so they all run during training
from stable_baselines3.common.evaluation import evaluate_policy  # Runs N episodes and returns mean/std reward
from stable_baselines3.common.env_util import make_vec_env       # Creates multiple parallel envs for faster data collection
from stable_baselines3.common.utils import get_linear_fn         # Returns a function that linearly decays a value over training

Create Project Folders

In [ ]:
# PROJECT FOLDERS
# Keeping logs, models, and videos in separate subfolders prevents clutter
# and makes it easy to reload checkpoints or graphs later.

BASE_DIR = "bipedalwalker_project"                        # Root folder for everything related to this run
LOG_DIR = os.path.join(BASE_DIR, "logs")                  # Where Monitor CSVs (episode rewards) are saved
MODEL_DIR = os.path.join(BASE_DIR, "models")              # Where checkpoint models are saved every N steps
BEST_MODEL_DIR = os.path.join(BASE_DIR, "best_model")     # Where EvalCallback saves the highest-scoring model seen so far
VIDEO_DIR = os.path.join(BASE_DIR, "videos")              # Reserved for recording video of the agent (not used in training)

os.makedirs(LOG_DIR, exist_ok=True)        # Create logs/ — exist_ok=True means no error if it already exists
os.makedirs(MODEL_DIR, exist_ok=True)      # Create models/
os.makedirs(BEST_MODEL_DIR, exist_ok=True) # Create best_model/
os.makedirs(VIDEO_DIR, exist_ok=True)      # Create videos/

print("Folders created successfully.")     # Confirm all directories are ready before training starts

Create New Eval Evn

In [ ]:
# ============================================================
# ENVIRONMENT SETUP
# ============================================================
# BipedalWalker-v3 is the environment the agent will learn in.
# We use:
#   - a vectorized training environment for faster learning
#   - a separate evaluation environment for testing progress
#   - monitor logs so we can graph rewards with Matplotlib
# ============================================================

ENV_ID = "BipedalWalker-v3"  # The Gymnasium environment ID — a 2D physics walker with 24 observations and 4 continuous actions
N_ENVS = 8                   # Run 8 environments in parallel — this multiplies how fast we collect experience

# make_vec_env creates N_ENVS copies of the environment running simultaneously.
# Each copy records its episode rewards to a separate .monitor.csv file in LOG_DIR.
# More parallel envs = more data per update = faster wall-clock training time.
train_env = make_vec_env(
    ENV_ID,              # Which environment to create
    n_envs=N_ENVS,       # How many parallel copies to run
    monitor_dir=LOG_DIR  # Directory where each env writes its reward log CSV
)

# The eval environment is a single, unwrapped environment used only for periodic testing.
# Monitor wraps it so evaluation rewards are also logged to CSV.
# It must be separate from train_env so evaluation episodes don't contaminate training data.
eval_env = Monitor(gym.make(ENV_ID))

print("Training and evaluation environments created.")

Wrap/ Log Env

Def PPO Hyperparameters 

In [ ]:
# ============================================================
# PPO MODEL SETUP
# ============================================================
# Tuned settings for better BipedalWalker performance:
#   - Larger network (256x256) for better policy capacity
#   - Bigger batch size to use rollout data more efficiently
#   - Linear LR decay for stable fine-tuning late in training
#   - Lower entropy coef to focus on exploitation over exploration
# ============================================================

model = PPO(
    policy="MlpPolicy",           # Use a standard fully-connected neural network for both policy and value function
    env=train_env,                # The vectorized training environment created above
    verbose=1,                    # Print training progress (loss, reward stats) to the console during training
    learning_rate=get_linear_fn(  # Learning rate decays linearly instead of staying fixed
        3e-4,                     #   Start value: 3e-4 (standard PPO starting LR — aggressive enough to learn fast early)
        1e-5,                     #   End value:   1e-5 (small LR near the end prevents overshooting a good policy)
        1.0                       #   Fraction of training over which the decay happens (1.0 = full training run)
    ),
    n_steps=2048,                 # Steps collected per environment per update — total rollout = n_steps * N_ENVS = 16,384 steps
    batch_size=256,               # Mini-batch size for gradient updates — larger batches give more stable gradient estimates
    n_epochs=10,                  # Number of times we reuse each rollout batch for gradient updates before collecting new data
    gamma=0.99,                   # Discount factor — rewards 100 steps in the future are worth 0.99^100 ≈ 37% of immediate reward
    gae_lambda=0.95,              # GAE (Generalized Advantage Estimation) smoothing — balances bias vs. variance in advantage estimates
    clip_range=0.2,               # PPO clipping threshold — limits how much the policy can change in a single update (prevents instability)
    ent_coef=0.001,               # Entropy bonus coefficient — small value encourages slight exploration without preventing the agent from committing
    policy_kwargs=dict(           # Custom neural network architecture
        net_arch=[dict(           # Separate networks for policy (pi) and value function (vf)
            pi=[256, 256],        #   Policy network: two hidden layers of 256 neurons each
            vf=[256, 256]         #   Value network: two hidden layers of 256 neurons each (default is only 64x64)
        )]
    ),
    tensorboard_log=os.path.join(BASE_DIR, "tensorboard")  # Save TensorBoard logs for visualizing loss and reward curves in real time
)

print("PPO model created.")

Create Callbacks and checkpoints 

In [ ]:
# ============================================================
# CALLBACKS
# ============================================================
# CheckpointCallback:
#   Saves the model every fixed number of steps.
#
# EvalCallback:
#   Tests the agent regularly on the separate eval environment.
#   Saves the best model automatically.
# ============================================================

checkpoint_callback = CheckpointCallback(
    save_freq=50_000 // N_ENVS,  # Save every 50,000 environment steps total. Divide by N_ENVS because SB3 counts steps per env, not total.
    save_path=MODEL_DIR,          # Folder where checkpoint .zip files are written
    name_prefix="ppo_bipedalwalker_checkpoint"  # Checkpoint files will be named like: ppo_bipedalwalker_checkpoint_50000_steps.zip
)

eval_callback = EvalCallback(
    eval_env,                          # The separate evaluation environment (not the training envs)
    best_model_save_path=BEST_MODEL_DIR,  # Save best_model.zip here whenever a new best mean reward is achieved
    log_path=BEST_MODEL_DIR,           # Also write evaluation results (evaluations.npz) to this folder
    eval_freq=25_000 // N_ENVS,        # Evaluate every 25,000 total steps — more frequent than checkpoints to catch the peak model
    deterministic=True,                # Use the greedy (no randomness) policy during evaluation for consistent scores
    render=False                       # Don't open a render window during training — faster and headless-server compatible
)

# CallbackList lets multiple callbacks run together during training.
# SB3 will call both checkpoint_callback and eval_callback at the appropriate steps.
callback = CallbackList([checkpoint_callback, eval_callback])

print("Callbacks ready.")

TRAIN

In [ ]:
# ============================================================
# TRAINING
# ============================================================
# 5M timesteps gives the agent enough experience to walk
# consistently and score above 300 (the "solved" threshold).
# ============================================================

TOTAL_TIMESTEPS = 5_000_000  # Total environment steps across all parallel envs. 5M is a common target for BipedalWalker.

model.learn(
    total_timesteps=TOTAL_TIMESTEPS,  # How long to train — the model will collect rollouts and update until this many steps are done
    callback=callback,                # The CallbackList containing checkpoint + eval callbacks — they fire automatically at the right intervals
    progress_bar=True                 # Show a tqdm progress bar so we can track how far through training we are
)

FINAL_MODEL_PATH = os.path.join(MODEL_DIR, "ppo_bipedalwalker_final")  # Build the file path for the final saved model (no .zip extension needed — SB3 adds it)
model.save(FINAL_MODEL_PATH)  # Save the model weights and hyperparameters to disk so they can be reloaded later without retraining

print(f"Training complete. Final model saved to: {FINAL_MODEL_PATH}")

Save CheckPoints 

Eval Periodically

In [ ]:
# ============================================================
# EVALUATE FINAL MODEL
# ============================================================
# This checks how well the final trained agent performs.
# ============================================================

# evaluate_policy runs the agent for n_eval_episodes complete episodes and returns statistics.
# deterministic=True means we use the greedy policy (no random sampling) for fair, repeatable results.
mean_reward, std_reward = evaluate_policy(
    model,              # The final model from the end of training
    eval_env,           # The separate eval environment (not the training envs)
    n_eval_episodes=10, # Run 10 full episodes to get a reliable performance estimate
    deterministic=True  # Greedy policy — pick the highest-probability action every step
)

print(f"Final model mean reward: {mean_reward:.2f}")  # Average reward over 10 episodes — higher is better (300+ is "solved")
print(f"Final model standard deviation: {std_reward:.2f}")  # Spread of rewards — high std means inconsistent performance

In [ ]:
# ============================================================
# LOAD AND EVALUATE THE BEST MODEL
# ============================================================
# Sometimes the best saved model performs better than the
# final model, depending on how training went near the end.
# ============================================================

best_model_path = os.path.join(BEST_MODEL_DIR, "best_model.zip")  # Path to the checkpoint EvalCallback saved when it saw the highest mean reward

best_model = PPO.load(best_model_path)  # Load the best model weights from disk — no need to retrain

# Evaluate the best model the same way we evaluated the final model, so the scores are directly comparable
best_mean_reward, best_std_reward = evaluate_policy(
    best_model,         # The best mid-training checkpoint, not the final weights
    eval_env,           # Same eval environment for a fair comparison
    n_eval_episodes=10, # Same number of episodes for consistent statistics
    deterministic=True  # Greedy policy for repeatable results
)

print(f"Best model mean reward: {best_mean_reward:.2f}")   # Compare this to the final model score above
print(f"Best model standard deviation: {best_std_reward:.2f}")  # If std is lower here, the best model is also more consistent

Plot results

In [ ]:
# ============================================================
# LOAD MONITOR LOG FILES
# ============================================================
# The Monitor wrapper saves episode rewards to .monitor.csv files.
# We load them all and combine them into one DataFrame so we can
# graph training progress with Matplotlib.
# ============================================================

def load_monitor_files(log_dir):
    csv_files = glob.glob(os.path.join(log_dir, "*.monitor.csv"))  # Find all monitor CSV files in the log directory (one per parallel env)
    dataframes = []  # Empty list to collect individual DataFrames before combining them

    for file in csv_files:  # Loop over every monitor file found
        df = pd.read_csv(file, skiprows=1)  # Load the CSV — skip row 0 which contains metadata (timestamp, env info), not actual data
        dataframes.append(df)  # Add this env's episode data to the list

    if not dataframes:  # If the list is still empty, no CSV files were found — raise a clear error instead of silently failing
        raise FileNotFoundError("No monitor files found in the log directory.")

    combined_df = pd.concat(dataframes, ignore_index=True)  # Stack all per-env DataFrames into one big DataFrame, resetting the row index
    return combined_df  # Return the combined DataFrame with columns: r (reward), l (length), t (timestamp)

monitor_df = load_monitor_files(LOG_DIR)  # Call the function to load and combine all monitor logs from training

print("Monitor logs loaded successfully.")
print(monitor_df.head())  # Preview the first few rows to confirm the data loaded correctly (columns: r, l, t)

In [ ]:
# ============================================================
# PLOT RAW EPISODE REWARDS
# ============================================================
# This shows the reward from each episode.
# Raw rewards are usually noisy in reinforcement learning.
# ============================================================

plt.figure(figsize=(12, 6))      # Create a new figure — 12 inches wide, 6 inches tall for a readable plot
plt.plot(monitor_df["r"].values) # Plot every episode's raw reward ("r" column from monitor CSV) in episode order
plt.xlabel("Episode")            # X-axis label — each point is one completed episode
plt.ylabel("Reward")             # Y-axis label — the total reward accumulated over that episode
plt.title("BipedalWalker Training Reward per Episode")  # Title to identify this graph
plt.grid(True)                   # Add a light grid to make it easier to read values off the chart
plt.show()                       # Render and display the plot inline in the notebook

In [ ]:
# ============================================================
# SMOOTH REWARDS
# ============================================================
# A rolling average helps make the learning trend easier to see.
# ============================================================

window = 50  # Window size for the rolling mean — average each episode's reward over the surrounding 50 episodes to reduce noise

# rolling(window=window).mean() computes a sliding window average:
#   For each episode i, it takes the mean of episodes [i-49 ... i]
#   The first 49 values will be NaN because there aren't enough prior episodes to fill the window yet
monitor_df["reward_smooth"] = monitor_df["r"].rolling(window=window).mean()  # Add the smoothed reward as a new column in the DataFrame

print(monitor_df[["r", "reward_smooth"]].head(10))  # Preview raw vs. smoothed values side-by-side to confirm the rolling average is working

In [ ]:
# ============================================================
# PLOT SMOOTHED REWARDS
# ============================================================
# This graph is much easier to use in a thesis because it shows
# the overall reward trend more clearly.
# ============================================================

plt.figure(figsize=(12, 6))                          # New figure — same dimensions as the raw reward plot for easy comparison
plt.plot(monitor_df["reward_smooth"].values)         # Plot only the rolling mean — noise is filtered out, trend is clear
plt.xlabel("Episode")                                # X-axis: episode number
plt.ylabel(f"Reward (Rolling Mean, window={window})") # Y-axis label includes the window size so the graph is self-documenting
plt.title("Smoothed BipedalWalker Training Reward")  # Title to distinguish this from the raw reward plot
plt.grid(True)                                       # Grid for readability
plt.show()                                           # Display the plot

In [ ]:
# ============================================================
# PLOT RAW + SMOOTHED REWARDS TOGETHER
# ============================================================
# This is the best overall graph because it shows both:
#   - the noisy episode-by-episode rewards
#   - the smoother learning trend
# ============================================================

plt.figure(figsize=(12, 6))  # New figure, same size as previous plots
plt.plot(monitor_df["r"].values, alpha=0.3, label="Raw Reward")
# alpha=0.3 makes the raw reward line semi-transparent so it doesn't visually overpower the smooth line
# This gives context for how noisy training actually is without hiding the trend

plt.plot(monitor_df["reward_smooth"].values, linewidth=2, label="Smoothed Reward")
# linewidth=2 makes the smooth line thicker and more prominent — it's the main thing we want the viewer to read

plt.xlabel("Episode")    # X-axis: each episode from training
plt.ylabel("Reward")     # Y-axis: episode reward value
plt.title("BipedalWalker PPO Learning Curve")  # Title identifying this as the main learning curve graph
plt.legend()             # Show a legend distinguishing the raw and smoothed lines
plt.grid(True)           # Grid for easier value reading
plt.show()               # Render and display

In [ ]:
# ============================================================
# REWARD VS TIMESTEPS
# ============================================================
# This creates a cumulative timestep column so you can graph
# reward against total environment steps instead of episode number.
# ============================================================

monitor_df["cumulative_timesteps"] = monitor_df["l"].cumsum()
# "l" column = episode length (number of steps taken in that episode)
# .cumsum() computes the running total of steps across all episodes
# This gives each episode an x-position on the "total training steps" axis instead of just episode number

plt.figure(figsize=(12, 6))  # New figure for this timestep-based view
plt.plot(monitor_df["cumulative_timesteps"], monitor_df["r"], alpha=0.3, label="Raw Reward")
# X: total steps elapsed so far | Y: raw episode reward | alpha=0.3 for transparency (same style as previous plot)

plt.plot(monitor_df["cumulative_timesteps"], monitor_df["reward_smooth"], linewidth=2, label="Smoothed Reward")
# X: total steps elapsed so far | Y: smoothed reward — shows the learning trend relative to compute budget

plt.xlabel("Timesteps")   # X-axis now shows actual training steps, not episode count — more meaningful for comparing algorithms
plt.ylabel("Reward")      # Y-axis: episode reward
plt.title("BipedalWalker Reward vs Timesteps")  # Title distinguishes this from the episode-indexed plots
plt.legend()              # Legend to label raw vs. smoothed lines
plt.grid(True)            # Grid for readability
plt.show()                # Display the plot

Record Final Run

In [ ]:
# ============================================================
# WATCH THE TRAINED AGENT
# ============================================================
# This opens a render window so you can visually inspect whether
# the walker has actually learned to move well.
# ============================================================

watch_env = gym.make(ENV_ID, render_mode="human")  # Create a fresh environment with render_mode="human" — opens a GUI window showing the walker
obs, info = watch_env.reset()  # Reset the environment to the start state; obs is the initial 24-dimensional observation vector

for step in range(2000):  # Run for up to 2000 steps — one BipedalWalker episode is at most 1600 steps, so this covers at least one full episode
    action, _states = model.predict(obs, deterministic=True)
    # model.predict — run the policy network on the current observation to get the next action
    # deterministic=True — use the greedy action (highest probability) instead of sampling randomly
    # _states is only used by recurrent policies (LSTM); for MlpPolicy it's None and can be ignored

    obs, reward, terminated, truncated, info = watch_env.step(action)
    # Send the chosen action to the environment
    # obs       — new observation after taking the action
    # reward    — reward received for this step
    # terminated — True if the agent fell over or reached the goal (episode is done naturally)
    # truncated  — True if the episode hit the time limit (1600 steps) without a natural end
    # info       — extra diagnostic data from the environment (usually empty for BipedalWalker)

    if terminated or truncated:  # If the episode ended for any reason, reset and start a new one
        obs, info = watch_env.reset()  # Reset returns a fresh observation and info dict for the new episode

watch_env.close()  # Close the render window and free the environment resources when we're done watching